# Strategic Annual Fire Severity Model — Governorate Level

This notebook documents and rebuilds the **strategic annual wildfire severity model**.

## Purpose

The model answers a yearly planning question:

> For each Tunisian governorate and year, what is the expected annual wildfire severity?

It produces three linked outputs:

1. **Predicted risk class**: `Low`, `Medium`, or `High`
2. **Predicted annual fire count**
3. **Predicted annual burned area in hectares**

This is not a daily alert model. It is a strategic, governorate-level model for annual preparedness, prioritization, and long-term planning.

## Main improvements retained in this notebook

The final strategic workflow keeps the original model objective, but corrects the unstable annual severity target:

- The old governorate-normalized min/max score could classify very small annual outcomes as `High` in low-history governorates.
- The improved target uses a **global robust log severity score**:
  - 60% annual fire count
  - 40% annual burned area
- Satellite detection fields such as brightness, confidence, FRP, and satellite fire count are excluded from predictors.
- Lagged and rolling historical features are added causally, using only previous years.
- 2025 is used as an out-of-sample audit year, then the production model is retrained on the full available period.


## 1. Imports and configuration

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42

# The notebook supports both a flat repository layout and a data/ folder layout.
ROOT = Path.cwd()
CANDIDATE_DATA_PATHS = [
    ROOT / "refined_tunisia_wildfire_dataset.csv",
    ROOT / "refined_tunisia_wildfire_dataset(1).csv",
    ROOT / "data" / "refined_tunisia_wildfire_dataset.csv",
    Path("/mnt/data/refined_tunisia_wildfire_dataset(1).csv"),  # useful inside this workspace
]

DATA_PATH = next((p for p in CANDIDATE_DATA_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find refined_tunisia_wildfire_dataset.csv. "
        "Place it beside this notebook or in a data/ folder."
    )

OUTPUT_DIR = ROOT / "strategic_annual_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

TRAIN_YEARS_AUDIT = list(range(2015, 2025))
TEST_YEAR_AUDIT = 2025

LOW_THRESHOLD = 0.10
HIGH_THRESHOLD = 0.55

FIRE_WEIGHT = 0.60
AREA_WEIGHT = 0.40

LEAKAGE_COLUMNS = [
    "brightness",
    "total_frp",
    "confidence",
    "satellite_fire_count",
]
TARGET_COLUMNS = [
    "fire_count",
    "area_burned_ha",
    "risk_class",
    "risk_score",
    "strategic_severity_score",
    "strategic_risk_class",
]


## 2. Load and audit the annual governorate dataset

Each row represents one governorate-year. The model must preserve this unit of analysis: it should not be mixed with daily pixel rows.

In [2]:
raw = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Rows: {len(raw):,}")
print(f"Columns: {raw.shape[1]:,}")
display(raw.head())

summary = {
    "years": [int(raw["year"].min()), int(raw["year"].max())],
    "governorates": int(raw["gouvernorat"].nunique()),
    "rows_per_year": raw.groupby("year").size().to_dict(),
    "missing_by_column": raw.isna().sum().sort_values(ascending=False).head(12).to_dict(),
}
summary


Loaded: /mnt/data/refined_tunisia_wildfire_dataset(1).csv
Rows: 264
Columns: 19


,gouvernorat,year,fire_count,area_burned_ha,forest_area_ha,other_forest_area_ha,tavg,tmax,tmin,prcp,wspd,pres,NDVI,NDMI,NBR,brightness,total_frp,confidence,satellite_fire_count
0,Ariana,2015,10.0,4.608,1141.0,1522.0,23.136364,34.6,12.5,15.2,17.430000,1020.100000,0.092032,-0.069317,0.000806,324.809091,158.0,64.363636,11.0
1,Ariana,2016,15.0,36.580,1141.0,1522.0,21.453333,39.0,4.5,49.1,14.306667,1017.191071,0.090146,-0.077382,-0.001329,329.433333,851.2,66.125000,24.0
2,Ariana,2017,10.0,4.250,1141.0,1522.0,19.532880,42.7,-4.8,318.4,13.078474,1017.244904,0.095317,-0.061661,0.010830,321.384615,183.3,61.692308,13.0
3,Ariana,2018,9.0,35.875,1141.0,1522.0,19.576712,43.6,-6.5,491.3,12.968407,1012.401700,0.095169,-0.066270,0.004231,318.022222,127.6,62.333333,9.0
4,Ariana,2019,8.0,26.654,1141.0,1522.0,19.906085,44.1,4.0,530.5,13.939683,1015.589947,0.098508,-0.061998,0.009875,325.719231,812.4,66.230769,26.0


{'years': [2015, 2025],
 'governorates': 24,
 'rows_per_year': {2015: 24,
  2016: 24,
  2017: 24,
  2018: 24,
  2019: 24,
  2020: 24,
  2021: 24,
  2022: 24,
  2023: 24,
  2024: 24,
  2025: 24},
 'missing_by_column': {'gouvernorat': 0,
  'year': 0,
  'fire_count': 0,
  'area_burned_ha': 0,
  'forest_area_ha': 0,
  'other_forest_area_ha': 0,
  'tavg': 0,
  'tmax': 0,
  'tmin': 0,
  'prcp': 0,
  'wspd': 0,
  'pres': 0}}

## 3. Data-quality and leakage policy

The annual model predicts future severity from annual weather, forest/vegetation context, and historical governorate behavior. It should not use fields that directly summarize active-fire detections in the same target year.

Excluded detection/leakage fields:

- `brightness`
- `total_frp`
- `confidence`
- `satellite_fire_count`

The targets themselves are also excluded from predictors.

In [3]:
def basic_quality_cleaning(frame: pd.DataFrame) -> pd.DataFrame:
    """Apply conservative annual-level data-quality fixes."""
    df = frame.copy()

    # Remove same-year satellite-detection descriptors from predictors.
    df = df.drop(columns=[c for c in LEAKAGE_COLUMNS if c in df.columns], errors="ignore")

    # Physically implausible values are converted to missing and handled inside pipelines.
    if "tmin" in df.columns:
        df.loc[df["tmin"] < -20, "tmin"] = np.nan
    if "prcp" in df.columns:
        df.loc[df["prcp"] < 0, "prcp"] = np.nan

    return df

wf = basic_quality_cleaning(raw)
print("Remaining columns:")
print(wf.columns.tolist())


Remaining columns:
['gouvernorat', 'year', 'fire_count', 'area_burned_ha', 'forest_area_ha', 'other_forest_area_ha', 'tavg', 'tmax', 'tmin', 'prcp', 'wspd', 'pres', 'NDVI', 'NDMI', 'NBR']


## 4. Improved strategic severity target

The target class is derived from the two annual severity quantities:

- annual `fire_count`
- annual `area_burned_ha`

The improved target uses a robust global log transformation. This avoids the earlier issue where governorate-specific min/max scaling could overstate risk in low-history governorates.

Formula:

```text
strategic_severity_score =
    0.60 * robust_log_fire_count
  + 0.40 * robust_log_burned_area
```

Where each robust component is min/max scaled using the 5th and 95th percentiles of the training/audit reference period.

Class thresholds:

```text
Low    : score < 0.10
Medium : 0.10 <= score < 0.55
High   : score >= 0.55
```


In [4]:
def robust_log_scale(values: pd.Series, p05: float, p95: float) -> pd.Series:
    """Log-scale values and robustly map them into [0, 1]."""
    x = np.log1p(values.astype(float).clip(lower=0))
    denom = max(p95 - p05, 1e-9)
    return ((x - p05) / denom).clip(0, 1)

def fit_severity_thresholds(reference: pd.DataFrame) -> dict:
    """Fit robust scaling thresholds on a reference period."""
    return {
        "fire_count_log_p05": float(np.log1p(reference["fire_count"].clip(lower=0)).quantile(0.05)),
        "fire_count_log_p95": float(np.log1p(reference["fire_count"].clip(lower=0)).quantile(0.95)),
        "area_log_p05": float(np.log1p(reference["area_burned_ha"].clip(lower=0)).quantile(0.05)),
        "area_log_p95": float(np.log1p(reference["area_burned_ha"].clip(lower=0)).quantile(0.95)),
        "low_threshold": LOW_THRESHOLD,
        "high_threshold": HIGH_THRESHOLD,
    }

def apply_severity_target(frame: pd.DataFrame, thresholds: dict) -> pd.DataFrame:
    """Create robust severity score and Low/Medium/High class."""
    df = frame.copy()
    count_scaled = robust_log_scale(
        df["fire_count"],
        thresholds["fire_count_log_p05"],
        thresholds["fire_count_log_p95"],
    )
    area_scaled = robust_log_scale(
        df["area_burned_ha"],
        thresholds["area_log_p05"],
        thresholds["area_log_p95"],
    )
    df["strategic_severity_score"] = FIRE_WEIGHT * count_scaled + AREA_WEIGHT * area_scaled
    df["strategic_risk_class"] = np.select(
        [
            df["strategic_severity_score"] < LOW_THRESHOLD,
            df["strategic_severity_score"] >= HIGH_THRESHOLD,
        ],
        ["Low", "High"],
        default="Medium",
    )
    return df

# Audit target definition: fit the target scaling on 2015-2024, then apply to 2025.
thresholds_audit = fit_severity_thresholds(wf.loc[wf["year"].isin(TRAIN_YEARS_AUDIT)])
panel = apply_severity_target(wf, thresholds_audit)

panel[["gouvernorat", "year", "fire_count", "area_burned_ha", "strategic_severity_score", "strategic_risk_class"]].head()


,gouvernorat,year,fire_count,area_burned_ha,strategic_severity_score,strategic_risk_class
0,Ariana,2015,10.0,4.608,0.436053,Medium
1,Ariana,2016,15.0,36.580,0.596490,High
2,Ariana,2017,10.0,4.250,0.432324,Medium
3,Ariana,2018,9.0,35.875,0.529054,Medium
4,Ariana,2019,8.0,26.654,0.497910,Medium


## 5. Causal feature engineering

Annual features are engineered so that historical variables for year `t` use only years `< t`.

Feature groups:

1. **Static fuel capacity**
   - forest area
   - other forest area
   - total forest area
2. **Weather and dryness**
   - annual temperature, precipitation, wind, pressure
   - heat/drought and vegetation dryness indices
3. **Historical fire behavior**
   - lagged fire count and burned area
   - rolling 3-year means/maxima
   - expanding historical mean/max before the current year
   - density per 1,000 ha


In [5]:
def add_causal_features(frame: pd.DataFrame) -> pd.DataFrame:
    df = frame.sort_values(["gouvernorat", "year"]).copy()

    df["total_forest_area_ha"] = df.get("forest_area_ha", 0).fillna(0) + df.get("other_forest_area_ha", 0).fillna(0)
    df["forest_share"] = df.get("forest_area_ha", 0).fillna(0) / df["total_forest_area_ha"].replace(0, np.nan)
    df["log_forest_area"] = np.log1p(df.get("forest_area_ha", 0).clip(lower=0))
    df["log_other_forest_area"] = np.log1p(df.get("other_forest_area_ha", 0).clip(lower=0))
    df["log_total_forest_area"] = np.log1p(df["total_forest_area_ha"].clip(lower=0))

    if {"tmax", "tmin"}.issubset(df.columns):
        df["temp_range"] = df["tmax"] - df["tmin"]
    if {"tmax", "prcp"}.issubset(df.columns):
        df["heat_drought_index"] = df["tmax"] / (df["prcp"].fillna(0) + 1.0)
        df["dryness_index"] = df["tmax"] - np.log1p(df["prcp"].fillna(0))
    if "NDMI" in df.columns:
        df["veg_dryness"] = -df["NDMI"]
    df["year_idx"] = df["year"] - df["year"].min()

    group = df.groupby("gouvernorat", group_keys=False)

    for target in ["fire_count", "area_burned_ha"]:
        for lag in [1, 2, 3]:
            df[f"{target}_lag{lag}"] = group[target].shift(lag)

        shifted = group[target].shift(1)
        df[f"{target}_roll3_mean"] = (
            shifted.groupby(df["gouvernorat"])
            .rolling(3, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )
        df[f"{target}_roll3_max"] = (
            shifted.groupby(df["gouvernorat"])
            .rolling(3, min_periods=1)
            .max()
            .reset_index(level=0, drop=True)
        )
        df[f"{target}_hist_mean"] = group[target].expanding().mean().groupby(level=0).shift(1).reset_index(level=0, drop=True)
        df[f"{target}_hist_max"] = group[target].expanding().max().groupby(level=0).shift(1).reset_index(level=0, drop=True)

    # Log history features stabilize very skewed annual outcomes.
    for c in [
        "fire_count_lag1",
        "fire_count_roll3_mean",
        "fire_count_hist_max",
        "area_burned_ha_lag1",
        "area_burned_ha_roll3_mean",
        "area_burned_ha_hist_max",
    ]:
        if c in df.columns:
            df[f"log1p_{c}"] = np.log1p(df[c].clip(lower=0))

    denom = df["total_forest_area_ha"].replace(0, np.nan)
    df["fire_density_lag1_per_1000ha"] = 1000 * df["fire_count_lag1"] / denom
    df["area_density_lag1_per_1000ha"] = 1000 * df["area_burned_ha_lag1"] / denom

    return df

panel_fe = add_causal_features(panel)

candidate_feature_columns = [
    "gouvernorat",
    "forest_area_ha", "other_forest_area_ha",
    "tavg", "tmax", "tmin", "prcp", "wspd", "pres",
    "NDVI", "NDMI", "NBR",
    "total_forest_area_ha", "forest_share",
    "log_forest_area", "log_other_forest_area", "log_total_forest_area",
    "temp_range", "heat_drought_index", "dryness_index", "veg_dryness", "year_idx",
    "fire_count_lag1", "fire_count_lag2", "fire_count_lag3",
    "fire_count_roll3_mean", "fire_count_roll3_max",
    "fire_count_hist_mean", "fire_count_hist_max",
    "log1p_fire_count_lag1", "log1p_fire_count_roll3_mean", "log1p_fire_count_hist_max",
    "area_burned_ha_lag1", "area_burned_ha_lag2", "area_burned_ha_lag3",
    "area_burned_ha_roll3_mean", "area_burned_ha_roll3_max",
    "area_burned_ha_hist_mean", "area_burned_ha_hist_max",
    "log1p_area_burned_ha_lag1", "log1p_area_burned_ha_roll3_mean", "log1p_area_burned_ha_hist_max",
    "fire_density_lag1_per_1000ha", "area_density_lag1_per_1000ha",
]
FEATURE_COLUMNS = [c for c in candidate_feature_columns if c in panel_fe.columns]
CATEGORICAL_FEATURES = ["gouvernorat"]
NUMERIC_FEATURES = [c for c in FEATURE_COLUMNS if c not in CATEGORICAL_FEATURES]

print(f"Feature count: {len(FEATURE_COLUMNS)}")
print(FEATURE_COLUMNS)


Feature count: 44
['gouvernorat', 'forest_area_ha', 'other_forest_area_ha', 'tavg', 'tmax', 'tmin', 'prcp', 'wspd', 'pres', 'NDVI', 'NDMI', 'NBR', 'total_forest_area_ha', 'forest_share', 'log_forest_area', 'log_other_forest_area', 'log_total_forest_area', 'temp_range', 'heat_drought_index', 'dryness_index', 'veg_dryness', 'year_idx', 'fire_count_lag1', 'fire_count_lag2', 'fire_count_lag3', 'fire_count_roll3_mean', 'fire_count_roll3_max', 'fire_count_hist_mean', 'fire_count_hist_max', 'log1p_fire_count_lag1', 'log1p_fire_count_roll3_mean', 'log1p_fire_count_hist_max', 'area_burned_ha_lag1', 'area_burned_ha_lag2', 'area_burned_ha_lag3', 'area_burned_ha_roll3_mean', 'area_burned_ha_roll3_max', 'area_burned_ha_hist_mean', 'area_burned_ha_hist_max', 'log1p_area_burned_ha_lag1', 'log1p_area_burned_ha_roll3_mean', 'log1p_area_burned_ha_hist_max', 'fire_density_lag1_per_1000ha', 'area_density_lag1_per_1000ha']


## 6. Preprocessing and model definitions

The notebook uses three predictive components:

1. **Risk-class classifier**
   - Random forest classifier
   - Class balancing enabled
2. **Fire-count regressors**
   - Ensemble of random forest, extra trees, histogram gradient boosting, and ridge regression on `log1p(fire_count)`
3. **Burned-area regressors**
   - Same ensemble family on `log1p(area_burned_ha)`

The final count and area predictions are converted back with `expm1`, clipped at zero, and averaged.

A conservative historical-capacity floor is applied to fire-count predictions to reduce systematic under-alerting in outbreak years.

In [6]:
def make_one_hot_encoder():
    # sklearn changed sparse -> sparse_output in newer versions.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preprocessor(scale_numeric: bool = False) -> ColumnTransformer:
    if scale_numeric:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
    else:
        numeric_pipe = SimpleImputer(strategy="median")

    return ColumnTransformer(
        transformers=[
            ("cat", make_one_hot_encoder(), CATEGORICAL_FEATURES),
            ("num", numeric_pipe, NUMERIC_FEATURES),
        ],
        remainder="drop",
    )

def make_regression_models() -> dict:
    return {
        "random_forest_log": Pipeline([
            ("preprocess", make_preprocessor()),
            ("model", RandomForestRegressor(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=2,
                random_state=RANDOM_SEED,
            )),
        ]),
        "extra_trees_log": Pipeline([
            ("preprocess", make_preprocessor()),
            ("model", ExtraTreesRegressor(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=2,
                random_state=RANDOM_SEED,
            )),
        ]),
        "hist_gradient_log": Pipeline([
            ("preprocess", make_preprocessor()),
            ("model", HistGradientBoostingRegressor(
                max_iter=150,
                learning_rate=0.05,
                max_leaf_nodes=8,
                min_samples_leaf=10,
                l2_regularization=0.1,
                random_state=RANDOM_SEED,
            )),
        ]),
        "ridge_log": Pipeline([
            ("preprocess", make_preprocessor(scale_numeric=True)),
            ("model", Ridge(alpha=10.0)),
        ]),
    }

def make_classifier() -> Pipeline:
    return Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", RandomForestClassifier(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_SEED,
        )),
    ])

def fit_log_ensemble(models: dict, X: pd.DataFrame, y: pd.Series) -> dict:
    y_log = np.log1p(y.clip(lower=0))
    fitted = {}
    for name, model in models.items():
        fitted[name] = model.fit(X, y_log)
    return fitted

def predict_log_ensemble(models: dict, X: pd.DataFrame) -> np.ndarray:
    preds = []
    for model in models.values():
        preds.append(np.expm1(model.predict(X)))
    pred = np.mean(np.vstack(preds), axis=0)
    return np.clip(pred, 0, None)

def apply_count_capacity_floor(frame: pd.DataFrame, raw_prediction: np.ndarray) -> np.ndarray:
    """Conservative guardrail for strategic annual count underprediction.

    Annual fire-count outbreaks can exceed what the regressors expect from smooth annual
    weather/fuel features. This floor does not replace the model. It only prevents
    severe under-alerting in governorates with substantial prior fire history.
    """
    pred = np.asarray(raw_prediction, dtype=float).copy()
    hist_max = frame["fire_count_hist_max"].fillna(0).to_numpy(dtype=float)
    hist_mean = frame["fire_count_hist_mean"].fillna(0).to_numpy(dtype=float)

    floor = np.where(
        hist_max >= 30,
        0.65 * hist_max,
        np.where(hist_max >= 10, 0.50 * hist_max, 0.0),
    )
    floor = np.maximum(floor, 1.25 * hist_mean)
    return np.maximum(pred, floor)


## 7. 2025 out-of-sample audit

The audit split is:

- Train: **2015–2024**
- Test: **2025**

This directly evaluates whether the improved strategic model handles the problematic 2025 predictions better.

In [7]:
train_df = panel_fe.loc[panel_fe["year"].isin(TRAIN_YEARS_AUDIT)].copy()
test_df = panel_fe.loc[panel_fe["year"] == TEST_YEAR_AUDIT].copy()

X_train = train_df[FEATURE_COLUMNS]
X_test = test_df[FEATURE_COLUMNS]

# Regressors for count and burned area.
count_models = fit_log_ensemble(make_regression_models(), X_train, train_df["fire_count"])
area_models = fit_log_ensemble(make_regression_models(), X_train, train_df["area_burned_ha"])

count_pred_raw = predict_log_ensemble(count_models, X_test)
count_pred = apply_count_capacity_floor(test_df, count_pred_raw)
area_pred = predict_log_ensemble(area_models, X_test)

# Strategic class classifier.
classifier = make_classifier()
classifier.fit(X_train, train_df["strategic_risk_class"])
class_pred = classifier.predict(X_test)

audit_metrics = {
    "fire_count_mae": float(mean_absolute_error(test_df["fire_count"], count_pred)),
    "fire_count_rmse": float(np.sqrt(mean_squared_error(test_df["fire_count"], count_pred))),
    "area_burned_mae": float(mean_absolute_error(test_df["area_burned_ha"], area_pred)),
    "area_burned_rmse": float(np.sqrt(mean_squared_error(test_df["area_burned_ha"], area_pred))),
    "risk_class_accuracy": float(accuracy_score(test_df["strategic_risk_class"], class_pred)),
    "risk_class_macro_f1": float(f1_score(test_df["strategic_risk_class"], class_pred, average="macro", zero_division=0)),
}

print(json.dumps(audit_metrics, indent=2))
print()
print(classification_report(test_df["strategic_risk_class"], class_pred, zero_division=0))


{
  "fire_count_mae": 26.300852993689585,
  "fire_count_rmse": 44.56539474706773,
  "area_burned_mae": 154.6898617350241,
  "area_burned_rmse": 370.5876247203517,
  "risk_class_accuracy": 0.5,
  "risk_class_macro_f1": 0.46356275303643724
}

              precision    recall  f1-score   support

        High       1.00      0.58      0.74        12
         Low       0.33      1.00      0.50         4
      Medium       0.20      0.12      0.15         8

    accuracy                           0.50        24
   macro avg       0.51      0.57      0.46        24
weighted avg       0.62      0.50      0.50        24



## 8. Save 2025 predictions and production model bundle

After the 2025 audit, the production bundle is retrained on all available annual records. This is appropriate for future strategic predictions after 2025 because the audit has already been completed.

In [8]:
# Attach audit predictions for 2025.
audit_out = test_df.copy()
audit_out["predicted_risk_class"] = class_pred
audit_out["predicted_fire_count"] = count_pred
audit_out["predicted_area_burned_ha"] = area_pred

proba = classifier.predict_proba(X_test)
for i, class_name in enumerate(classifier.classes_):
    audit_out[f"prob_{class_name}"] = proba[:, i]

audit_path = OUTPUT_DIR / "strategic_annual_2025_audit_predictions.csv"
audit_out.to_csv(audit_path, index=False)
print(f"Saved audit predictions: {audit_path}")

# Production thresholds and target labels are refit on all available history.
thresholds_production = fit_severity_thresholds(wf)
production_panel = apply_severity_target(wf, thresholds_production)
production_panel = add_causal_features(production_panel)

X_all = production_panel[FEATURE_COLUMNS]
count_models_prod = fit_log_ensemble(make_regression_models(), X_all, production_panel["fire_count"])
area_models_prod = fit_log_ensemble(make_regression_models(), X_all, production_panel["area_burned_ha"])

classifier_prod = make_classifier()
classifier_prod.fit(X_all, production_panel["strategic_risk_class"])

production_class_pred = classifier_prod.predict(X_all)
production_count_pred = apply_count_capacity_floor(
    production_panel,
    predict_log_ensemble(count_models_prod, X_all),
)
production_area_pred = predict_log_ensemble(area_models_prod, X_all)

predictions = production_panel.copy()
predictions["predicted_risk_class"] = production_class_pred
predictions["predicted_fire_count"] = production_count_pred
predictions["predicted_area_burned_ha"] = production_area_pred
predictions["model_version"] = "strategic_annual_v2_robust_hybrid"

for i, class_name in enumerate(classifier_prod.classes_):
    predictions[f"prob_{class_name}"] = classifier_prod.predict_proba(X_all)[:, i]

predictions_path = OUTPUT_DIR / "annual_governorate_predictions_improved.csv"
predictions.to_csv(predictions_path, index=False)

bundle = {
    "model_version": "strategic_annual_v2_robust_hybrid",
    "count_models": count_models_prod,
    "area_models": area_models_prod,
    "classifier": classifier_prod,
    "feature_columns": FEATURE_COLUMNS,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "thresholds": thresholds_production,
    "low_threshold": LOW_THRESHOLD,
    "high_threshold": HIGH_THRESHOLD,
    "class_order": ["Low", "Medium", "High"],
    "leakage_columns": LEAKAGE_COLUMNS,
    "train_years": sorted(production_panel["year"].unique().tolist()),
    "prediction_logic": "Strategic annual governorate severity model with robust log severity target.",
}
bundle_path = OUTPUT_DIR / "annual_governorate_model_bundle_improved.joblib"
joblib.dump(bundle, bundle_path)

metrics = {
    "model_version": bundle["model_version"],
    "evaluation_setup": "Audit train 2015-2024, test 2025; production retrained on all available annual years.",
    "risk_target_definition": {
        "score": "0.6 * robust_log_fire_count + 0.4 * robust_log_burned_area",
        "low_threshold": LOW_THRESHOLD,
        "high_threshold": HIGH_THRESHOLD,
    },
    "test_2025": audit_metrics,
    "confusion_matrix_2025": {
        "labels": ["Low", "Medium", "High"],
        "matrix": confusion_matrix(
            test_df["strategic_risk_class"],
            class_pred,
            labels=["Low", "Medium", "High"],
        ).tolist(),
    },
    "feature_columns": FEATURE_COLUMNS,
}
metrics_path = OUTPUT_DIR / "annual_governorate_metrics_improved.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(f"Saved production predictions: {predictions_path}")
print(f"Saved production bundle: {bundle_path}")
print(f"Saved metrics: {metrics_path}")


Saved audit predictions: /strategic_annual_outputs/strategic_annual_2025_audit_predictions.csv


Saved production predictions: /strategic_annual_outputs/annual_governorate_predictions_improved.csv
Saved production bundle: /strategic_annual_outputs/annual_governorate_model_bundle_improved.joblib
Saved metrics: /strategic_annual_outputs/annual_governorate_metrics_improved.json


## 9. How to use this model

Use this notebook when the task is **strategic annual planning**.

Typical outputs:

- annual `predicted_risk_class`
- annual `predicted_fire_count`
- annual `predicted_area_burned_ha`

Do not use this model for same-day pixel alerts. The operational pixel model is documented in the second notebook.
